In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import sqlite3
import random, re, hashlib

In [4]:
url = "https://api.tcgdex.net/v2/en/cards"

response_all_cards = requests.get(url, timeout=15)
response_all_cards.raise_for_status()

response_all_cards_json  = response_all_cards.json()


In [5]:
print("Total number of cards:", len(response_all_cards_json))
print("Sample card data:", response_all_cards_json[3])

Total number of cards: 23160
Sample card data: {'id': 'swsh9-001', 'localId': '001', 'name': 'Exeggcute', 'image': 'https://assets.tcgdex.net/en/swsh/swsh9/001'}


In [6]:
def get_card_details(card_id):
    url = f"https://api.tcgdex.net/v2/en/cards/{card_id}"
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    return response.json()
card_details = get_card_details(response_all_cards_json[3]['id'])
print("Card details:", card_details)

Card details: {'category': 'Pokemon', 'id': 'swsh9-001', 'image': 'https://assets.tcgdex.net/en/swsh/swsh9/001', 'localId': '001', 'name': 'Exeggcute', 'rarity': 'Common', 'set': {'cardCount': {'official': 172, 'total': 216}, 'id': 'swsh9', 'logo': 'https://assets.tcgdex.net/en/swsh/swsh9/logo', 'name': 'Brilliant Stars', 'symbol': 'https://assets.tcgdex.net/univ/swsh/swsh9/symbol'}, 'variants': {'firstEdition': False, 'holo': False, 'normal': True, 'reverse': True, 'wPromo': False}, 'variants_detailed': [{'type': 'normal', 'size': 'standard', 'variantId': 'generated'}, {'type': 'reverse', 'size': 'standard', 'variantId': 'generated'}], 'dexId': [102], 'hp': 50, 'types': ['Grass'], 'stage': 'Basic', 'attacks': [{'cost': ['Colorless'], 'name': 'Ram', 'damage': 10}, {'cost': ['Grass', 'Colorless'], 'name': 'Seed Bomb', 'damage': 20}], 'retreat': 1, 'regulationMark': 'F', 'legal': {'standard': False, 'expanded': True}, 'updated': '2025-08-16T20:39:55Z', 'pricing': {'cardmarket': {'updated

In [7]:
card_details["set"]

{'cardCount': {'official': 172, 'total': 216},
 'id': 'swsh9',
 'logo': 'https://assets.tcgdex.net/en/swsh/swsh9/logo',
 'name': 'Brilliant Stars',
 'symbol': 'https://assets.tcgdex.net/univ/swsh/swsh9/symbol'}

In [8]:
# JSON (dict) -> DataFrame "piatto" (1 riga)
df_card = pd.json_normalize(card_details, sep="_")
display(df_card)

,category,id,image,localId,name,rarity,variants_detailed,dexId,hp,types,...,pricing_cardmarket_avg1,pricing_cardmarket_avg7,pricing_cardmarket_avg30,pricing_cardmarket_avg-holo,pricing_cardmarket_low-holo,pricing_cardmarket_trend-holo,pricing_cardmarket_avg1-holo,pricing_cardmarket_avg7-holo,pricing_cardmarket_avg30-holo,pricing_tcgplayer
0,Pokemon,swsh9-001,https://assets.tcgdex.net/en/swsh/swsh9/001,001,Exeggcute,Common,"[{'type': 'normal', 'size': 'standard', 'varia...",[102],50,[Grass],...,0.02,0.03,0.03,0.16,0.02,0.09,0.05,0.19,0.16,None


In [9]:
all_df_cards = []
for card in tqdm(response_all_cards_json[:300]):
    try:
        details = get_card_details(card['id'])
        df_card = pd.json_normalize(details, sep="_")
        df_card["espansione_id"] = details["set"]["id"]
        df_card["espansione_nome"] = details["set"]["name"]
        if "image" not in details: 
            df_card["image"] = [None]
        if not details["pricing"]["cardmarket"]:
            df_card["pricing_cardmarket_low"] = [0]
        df_card = df_card[["id", "name","espansione_id", "espansione_nome", "pricing_cardmarket_low", "image"]]
        all_df_cards.append(df_card)
    except Exception as e:
        print(f"Error fetching details for card {card['id']}: {e}")
        #break

  1%|          | 2/300 [00:00<01:55,  2.59it/s]

Error fetching details for card exu-%3F: 404 Client Error: Not Found for url: https://api.tcgdex.net/v2/en/cards/exu-%3F


100%|██████████| 300/300 [01:51<00:00,  2.69it/s]


In [ ]:
conn = sqlite3.connect("../card_database.db")
cursor = conn.cursor()

df_all = pd.concat(all_df_cards, ignore_index=True)
df_all.to_sql(
    "DatabaseCards",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)

### Generate stock example

In [10]:
def generate_barcode(nome, espansione, condizione):
    def clean(s):
        return re.sub(r"[^A-Z0-9]", "", s.upper())

    # parte leggibile
    base = f"{clean(nome)[:4]}-{clean(espansione)[:3]}-{clean(condizione)[:2]}"

    # hash deterministico
    raw = f"{nome}|{espansione}|{condizione}".upper()
    short_hash = hashlib.md5(raw.encode()).hexdigest()[:6].upper()

    return f"{base}-{short_hash}"


generate_barcode("Pikachu V Full Art", "SWSH039", "Excellent")

'PIKA-SWS-EX-4BF6D1'

In [12]:
df_all = pd.concat(all_df_cards, ignore_index=True)
df_test = df_all.sample(15)[["id", "name","espansione_id", "espansione_nome"]].copy()
cards_condizioni = ["Mint", "Near Mint", "Excellent", "Good", "Light Played", "Played", "Poor"]
df_test["condizione"] = [random.choice(cards_condizioni) for _ in range(len(df_test))]
df_test["barcode"] = df_test.apply(lambda row: generate_barcode(row["name"], row["espansione_nome"], row["condizione"]), axis=1)
df_test["prezzo"] = [round(random.uniform(1, 100), 2) for _ in range(len(df_test))]
df_test["quantita_stock"] = [random.randint(1, 5) for _ in range(len(df_test))]
df_test["prezzo_acquisto"] = [round(random.uniform(0, 1) * prezzo, 2)  for prezzo in df_test["prezzo"]]
df_test

C:\Users\s.galati\AppData\Local\Temp\ipykernel_14776\903807934.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(all_df_cards, ignore_index=True)


,id,name,espansione_id,espansione_nome,condizione,barcode,prezzo,quantita_stock,prezzo_acquisto
28,xy3-1,Bellsprout,xy3,Furious Fists,Excellent,BELL-FUR-EX-AB64BC,67.79,1,50.61
113,ru1-1,Venusaur,ru1,Pokémon Rumble,Light Played,VENU-POK-LI-5B6E03,86.86,5,33.18
32,tk-dp-l-1,Geodude,tk-dp-l,DP trainer Kit (Lucario),Near Mint,GEOD-DPT-NE-E54B22,98.22,4,64.69
119,2017sm-1,Rowlet,2017sm,McDonald's Collection 2017,Light Played,ROWL-MCD-LI-8FA858,17.02,1,8.56
293,mep-002,Inteleon,mep,MEP Black Star Promos,Poor,INTE-MEP-PO-949679,74.56,5,46.66
140,ex8-1,Altaria,ex8,Deoxys,Poor,ALTA-DEO-PO-42F418,9.17,1,4.02
40,tk-ex-p-1,Beldum,tk-ex-p,EX trainer Kit 2 (Plusle),Poor,BELD-EXT-PO-C4AD47,80.74,2,59.65
261,det1-2,Ludicolo,det1,Detective Pikachu,Light Played,LUDI-DET-LI-03BE5E,42.04,4,0.20
265,sm115-2,Metapod,sm115,Hidden Fates,Good,META-HID-GO-5C5857,93.21,3,67.05
275,hgss3-2,Espeon,hgss3,Undaunted,Good,ESPE-UND-GO-C5082E,68.07,1,40.73


In [14]:
conn = sqlite3.connect("../pokemon.db")
cursor = conn.cursor()
df_test.to_sql(
    "stock",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)
conn.close()